# Tomahawk TF48: 48-field-period to 16-field-period MGRID conversion check

This notebook reads the supplied 48-period MAKEGRID/VMEC NetCDF grid, forms the 16-period representation by repeating each 7.5° source period three times, writes the converted MGRID, and checks the saved arrays plane by plane.

## Kaggle setup

1. Add a Kaggle Dataset containing `mgrid_tomahawk_tf48_16T_PERIODIC_FP64.nc` to this notebook's **Input** panel. The JSON verification manifest is already embedded below; it does not contain the field arrays.
2. If you also have `mgrid_tomahawk_tf48_16FIELD_16T_PERIODIC_FP64.nc`, add it too. The notebook will compare it with the newly generated output.
3. Run all cells. The converted grid and a machine-readable report will be written to `/kaggle/working/` for download from the notebook's Output panel.

The conversion changes only the field-period representation and toroidal plane count (`nfp: 48 → 16`, `kp: 32 → 96`). It does not interpolate or rescale the field values. This is an exact symmetry re-indexing check, not a 3-D MHD equilibrium or hardware qualification. The axis-field and ripple numbers below are carried forward from the supplied manifest; this notebook does not independently recalculate them.


In [ ]:
from pathlib import Path
import hashlib
import json
import re
import sys
import subprocess
from datetime import datetime, timezone

import numpy as np

try:
    from netCDF4 import Dataset
except ImportError:
    # Kaggle images normally include netCDF4. If this image does not, enable
    # notebook Internet or add netCDF4 to the Kaggle environment, then rerun.
    subprocess.check_call([sys.executable, "-m", "pip", "install", "netCDF4"])
    from netCDF4 import Dataset

MANIFEST = {
    "status": "PASS_EXACT_16_FIELD_PERIOD_REPRESENTATION",
    "field_baseline_T": 16.0,
    "physical_TF_coils": 48,
    "physical_coils_per_field_period": 3,
    "source_grid": {
        "name": "mgrid_tomahawk_tf48_16T_PERIODIC_FP64.nc",
        "nfp": 48,
        "kp": 32,
        "raw_coil_current_A": -125000.0,
    },
    "derived_grid": {
        "name": "mgrid_tomahawk_tf48_16FIELD_16T_PERIODIC_FP64.nc",
        "nfp": 16,
        "kp": 96,
        "radial_nodes": 201,
        "vertical_nodes": 201,
        "raw_coil_current_A": -125000.0,
    },
    "angular_spacing_deg": 0.234375,
    "full_torus_angular_planes": 1536,
    "source_centerline_coil_count": 48,
    "segments_per_physical_coil": 512,
    "maximum_centerline_spacing_error_rad": 3.6914915568786455e-15,
    "exact_plane_expansion_max_abs_error_T": 0.0,
    "axis_Bphi_T_from_source_verification": 16.000000098666238,
    "edge_ripple_percent_from_source_verification": 0.7556405502813015,
    "edge_ripple_limit_percent": 1.0,
    "method": "The independently verified FP64 nfp=48, kp=32 planes repeat three times over each 22.5-degree nfp=16 period. All three field components are copied without interpolation or amplitude changes.",
    "scope": "Exact symmetry re-indexing of the verified 48-coil vacuum-field grid; no additional MHD equilibrium or hardware qualification.",
}

# Optional overrides. Leave as None for automatic search under /kaggle/input.
SOURCE_PATH_OVERRIDE = None
EXISTING_DERIVED_PATH_OVERRIDE = None
OUTPUT_DIR = Path("/kaggle/working")
COMPARE_EXISTING_DERIVED_IF_FOUND = True

print("Manifest loaded:", MANIFEST["status"])
print("Source grid:", MANIFEST["source_grid"]["name"])


In [ ]:
def discover_input(filename, override=None):
    if override:
        path = Path(override)
        if not path.is_file():
            raise FileNotFoundError(f"Configured file does not exist: {path}")
        return path

    root = Path("/kaggle/input")
    matches = sorted(root.rglob(filename)) if root.exists() else []
    if len(matches) > 1:
        listing = "\n".join(str(p) for p in matches)
        raise RuntimeError(f"More than one {filename} was found. Set the corresponding *_PATH_OVERRIDE explicitly:\n{listing}")
    return matches[0] if matches else None

def sha256_file(path, block_size=1024 * 1024):
    digest = hashlib.sha256()
    with open(path, "rb") as stream:
        for block in iter(lambda: stream.read(block_size), b""):
            digest.update(block)
    return digest.hexdigest()

def scalar_number(ds, name):
    if name not in ds.variables:
        raise KeyError(f"Required MGRID metadata variable '{name}' is missing")
    data = np.asarray(ds.variables[name][...])
    if data.size != 1:
        raise ValueError(f"Expected scalar metadata variable '{name}', found shape {data.shape}")
    return float(data.reshape(-1)[0])

def field_variables(ds):
    pattern = re.compile(r"^(br|bp|bz)_(\d+)$", re.IGNORECASE)
    found = {}
    for name in ds.variables:
        match = pattern.match(name)
        if match:
            component, group = match.group(1).lower(), match.group(2)
            found.setdefault(group, {})[component] = name
    if not found:
        raise ValueError("No br_NNN/bp_NNN/bz_NNN MAKEGRID field variables were found")
    required = {"br", "bp", "bz"}
    for group, components in found.items():
        if set(components) != required:
            raise ValueError(f"Field group {group} does not contain all three components: {sorted(components)}")
    return found

def identify_phi_dimension(ds, expected_kp, groups):
    all_field_names = [name for components in groups.values() for name in components.values()]
    candidates = set()
    for name in all_field_names:
        var = ds.variables[name]
        if var.ndim != 3:
            raise ValueError(f"Expected a 3-D field variable, got {name}{var.shape}")
        for dim in var.dimensions:
            if len(ds.dimensions[dim]) == expected_kp:
                candidates.add(dim)
    preferred = [name for name in candidates if name.lower() == "phi"]
    if len(preferred) == 1:
        phi_dim = preferred[0]
    elif len(candidates) == 1:
        phi_dim = next(iter(candidates))
    else:
        raise ValueError(f"Could not uniquely identify the toroidal dimension of length {expected_kp}; candidates={sorted(candidates)}")

    for name in all_field_names:
        var = ds.variables[name]
        if phi_dim not in var.dimensions:
            raise ValueError(f"Toroidal dimension {phi_dim!r} is missing from field variable {name}")
        axis = var.dimensions.index(phi_dim)
        if var.shape[axis] != expected_kp:
            raise ValueError(f"Unexpected toroidal axis length for {name}: {var.shape[axis]}")
    return phi_dim

def preflight_source(path):
    with Dataset(path, "r") as ds:
        ds.set_auto_maskandscale(False)
        nfp = int(round(scalar_number(ds, "nfp")))
        kp = int(round(scalar_number(ds, "kp")))
        ir = int(round(scalar_number(ds, "ir")))
        jz = int(round(scalar_number(ds, "jz")))
        expected = MANIFEST["source_grid"]
        if (nfp, kp) != (expected["nfp"], expected["kp"]):
            raise AssertionError(f"Source metadata mismatch: found nfp={nfp}, kp={kp}; expected nfp={expected['nfp']}, kp={expected['kp']}")
        target = MANIFEST["derived_grid"]
        if (ir, jz) != (target["radial_nodes"], target["vertical_nodes"]):
            raise AssertionError(f"Source radial/vertical grid mismatch: ir={ir}, jz={jz}")

        groups = field_variables(ds)
        phi_dim = identify_phi_dimension(ds, kp, groups)
        for components in groups.values():
            for name in components.values():
                dtype = np.dtype(ds.variables[name].dtype)
                if dtype != np.dtype("float64"):
                    raise AssertionError(f"Expected FP64 field arrays; {name} has dtype {dtype}")

        if "raw_coil_cur" not in ds.variables:
            raise KeyError("Required MGRID variable 'raw_coil_cur' is missing")
        raw_currents = np.asarray(ds.variables["raw_coil_cur"][...], dtype=np.float64).reshape(-1)
        expected_current = float(expected["raw_coil_current_A"])
        if not np.any(raw_currents == expected_current):
            raise AssertionError(f"Expected raw_coil_cur to contain {expected_current:g} A; values found: {raw_currents.tolist()[:12]}")

        return {
            "nfp": nfp,
            "kp": kp,
            "ir": ir,
            "jz": jz,
            "phi_dimension": phi_dim,
            "field_groups": groups,
            "field_variable_count": sum(len(c) for c in groups.values()),
            "raw_currents": raw_currents,
            "data_model": ds.data_model,
        }

source_path = discover_input(MANIFEST["source_grid"]["name"], SOURCE_PATH_OVERRIDE)
if source_path is None:
    available = sorted(str(p) for p in Path("/kaggle/input").rglob("*.nc")) if Path("/kaggle/input").exists() else []
    raise FileNotFoundError(
        "Source MGRID not found. Add a Kaggle Dataset containing "
        f"{MANIFEST['source_grid']['name']} to this notebook's Input panel. "
        "The attached JSON is a manifest and does not contain the field arrays. "
        f"NetCDF files currently visible: {available[:20]}"
    )

source_info = preflight_source(source_path)
print(f"Source preflight PASS: {source_path}")
print(f"nfp={source_info['nfp']}, kp={source_info['kp']}, ir={source_info['ir']}, jz={source_info['jz']}")
print(f"Field variables: {source_info['field_variable_count']} across {len(source_info['field_groups'])} coil groups")
print(f"NetCDF format: {source_info['data_model']}")


In [ ]:
def toroidal_axis(var, phi_dim):
    if phi_dim not in var.dimensions:
        raise ValueError(f"{var.name} does not use the toroidal dimension {phi_dim!r}")
    return var.dimensions.index(phi_dim)

def expanded_field_data(var, phi_dim, source_kp, derived_kp):
    axis = toroidal_axis(var, phi_dim)
    data = np.asarray(var[...])
    if data.shape[axis] != source_kp:
        raise ValueError(f"{var.name} has {data.shape[axis]} source planes, expected {source_kp}")
    plane_indices = np.arange(derived_kp, dtype=np.int64) % source_kp
    return np.take(data, plane_indices, axis=axis)

def copy_attribute_set(source, destination):
    for attr in source.ncattrs():
        if attr != "_FillValue":
            destination.setncattr(attr, source.getncattr(attr))

def create_output_grid(source_path, output_path, source_info):
    target = MANIFEST["derived_grid"]
    source_kp = source_info["kp"]
    derived_kp = int(target["kp"])
    phi_dim = source_info["phi_dimension"]
    output_path.parent.mkdir(parents=True, exist_ok=True)

    with Dataset(source_path, "r") as src:
        src.set_auto_maskandscale(False)
        with Dataset(output_path, "w", format=src.data_model) as dst:
            # Keep source dimensions and replace only the toroidal plane count.
            for name, dimension in src.dimensions.items():
                if name == phi_dim and dimension.isunlimited():
                    raise ValueError("The toroidal MGRID dimension must be fixed length")
                length = derived_kp if name == phi_dim else (None if dimension.isunlimited() else len(dimension))
                dst.createDimension(name, length)

            copy_attribute_set(src, dst)
            field_name_set = {n for group in source_info["field_groups"].values() for n in group.values()}
            for name, src_var in src.variables.items():
                create_kwargs = {}
                if "_FillValue" in src_var.ncattrs():
                    create_kwargs["fill_value"] = src_var.getncattr("_FillValue")
                if src.data_model == "NETCDF4":
                    filters = src_var.filters()
                    for key in ("zlib", "complevel", "shuffle", "fletcher32"):
                        if key in filters:
                            create_kwargs[key] = filters[key]
                    chunks = src_var.chunking()
                    if isinstance(chunks, (list, tuple)):
                        create_kwargs["chunksizes"] = tuple(chunks)
                    try:
                        endian = src_var.endian()
                        if endian in ("little", "big"):
                            create_kwargs["endian"] = endian
                    except Exception:
                        pass

                dst_var = dst.createVariable(name, src_var.datatype, src_var.dimensions, **create_kwargs)
                copy_attribute_set(src_var, dst_var)

                lower_name = name.lower()
                if name in field_name_set:
                    dst_var[...] = expanded_field_data(src_var, phi_dim, source_kp, derived_kp)
                elif lower_name in ("nfp", "kp"):
                    value = target[lower_name]
                    if int(np.prod(src_var.shape, dtype=np.int64)) != 1:
                        raise ValueError(f"Expected one metadata value in {name}, found shape {src_var.shape}")
                    if src_var.ndim == 0:
                        dst_var.assignValue(value)
                    else:
                        metadata = np.asarray(src_var[...]).copy()
                        metadata.reshape(-1)[0] = value
                        dst_var[...] = metadata
                else:
                    dst_var[...] = src_var[...]

    return output_path

def compare_grid_fields(source_path, candidate_path, source_info, label):
    target = MANIFEST["derived_grid"]
    maximum_absolute_error = 0.0
    compared = 0
    with Dataset(source_path, "r") as src, Dataset(candidate_path, "r") as candidate:
        src.set_auto_maskandscale(False)
        candidate.set_auto_maskandscale(False)

        for key in ("nfp", "kp", "ir", "jz"):
            actual = int(round(scalar_number(candidate, key)))
            expected = int(target[key]) if key in target else source_info[key]
            if actual != expected:
                raise AssertionError(f"{label} metadata {key}={actual}; expected {expected}")

        phi_dim = source_info["phi_dimension"]
        if phi_dim not in candidate.dimensions or len(candidate.dimensions[phi_dim]) != target["kp"]:
            raise AssertionError(f"{label} toroidal dimension is not {target['kp']} planes")

        source_currents = np.asarray(src.variables["raw_coil_cur"][...])
        candidate_currents = np.asarray(candidate.variables["raw_coil_cur"][...])
        if not np.array_equal(source_currents, candidate_currents):
            raise AssertionError(f"{label} changed raw_coil_cur")

        for components in source_info["field_groups"].values():
            for source_name in components.values():
                if source_name not in candidate.variables:
                    raise AssertionError(f"{label} is missing field variable {source_name}")
                expected_data = expanded_field_data(src.variables[source_name], phi_dim, source_info["kp"], target["kp"])
                actual_data = np.asarray(candidate.variables[source_name][...])
                if actual_data.shape != expected_data.shape:
                    raise AssertionError(f"{label} shape mismatch for {source_name}: {actual_data.shape} vs {expected_data.shape}")
                difference = np.abs(actual_data - expected_data)
                variable_error = float(np.max(difference)) if difference.size else 0.0
                maximum_absolute_error = max(maximum_absolute_error, variable_error)
                if not np.array_equal(actual_data, expected_data):
                    raise AssertionError(f"{label} field mismatch in {source_name}; max abs error={variable_error:.17g} T")
                compared += 1

    return {"field_variables_compared": compared, "max_abs_error_T": maximum_absolute_error}

output_path = OUTPUT_DIR / MANIFEST["derived_grid"]["name"]
existing_path = discover_input(MANIFEST["derived_grid"]["name"], EXISTING_DERIVED_PATH_OVERRIDE) if COMPARE_EXISTING_DERIVED_IF_FOUND else None

create_output_grid(source_path, output_path, source_info)
generated_check = compare_grid_fields(source_path, output_path, source_info, "Generated output")
print(f"Generated-grid exact comparison PASS: {generated_check['field_variables_compared']} field arrays; max error={generated_check['max_abs_error_T']:.1f} T")

existing_check = None
if existing_path is not None:
    existing_check = compare_grid_fields(source_path, existing_path, source_info, "Existing derived grid")
    print(f"Existing derived-grid comparison PASS: {existing_check['field_variables_compared']} field arrays; max error={existing_check['max_abs_error_T']:.1f} T")
else:
    print("No existing derived grid was found; generated output was checked directly against the source expansion.")


In [ ]:
source_planes_full_torus = source_info["nfp"] * source_info["kp"]
derived_planes_full_torus = MANIFEST["derived_grid"]["nfp"] * MANIFEST["derived_grid"]["kp"]
source_spacing_deg = 360.0 / source_planes_full_torus
derived_spacing_deg = 360.0 / derived_planes_full_torus

if source_planes_full_torus != MANIFEST["full_torus_angular_planes"]:
    raise AssertionError(f"Source full-torus plane count {source_planes_full_torus} does not match manifest")
if derived_planes_full_torus != MANIFEST["full_torus_angular_planes"]:
    raise AssertionError(f"Derived full-torus plane count {derived_planes_full_torus} does not match manifest")
if source_spacing_deg != MANIFEST["angular_spacing_deg"] or derived_spacing_deg != MANIFEST["angular_spacing_deg"]:
    raise AssertionError("Angular spacing does not match the manifest")

report = {
    "status": "PASS_EXACT_16_FIELD_PERIOD_REPRESENTATION",
    "run_timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "source_file": str(source_path),
    "source_sha256": sha256_file(source_path),
    "generated_derived_file": str(output_path),
    "generated_derived_sha256": sha256_file(output_path),
    "source_metadata": {
        "nfp": source_info["nfp"],
        "kp": source_info["kp"],
        "ir": source_info["ir"],
        "jz": source_info["jz"],
        "coil_group_count": len(source_info["field_groups"]),
        "field_variable_count": source_info["field_variable_count"],
        "raw_coil_cur_A": source_info["raw_currents"].tolist(),
        "netcdf_data_model": source_info["data_model"],
    },
    "derived_metadata": {
        "nfp": MANIFEST["derived_grid"]["nfp"],
        "kp": MANIFEST["derived_grid"]["kp"],
        "ir": source_info["ir"],
        "jz": source_info["jz"],
        "full_torus_angular_planes": derived_planes_full_torus,
        "angular_spacing_deg": derived_spacing_deg,
    },
    "computed_checks": {
        "generated_file_field_array_comparison": generated_check,
        "existing_derived_file_comparison": existing_check,
        "all_field_components_compared_exactly": True,
        "field_arrays_rescaled_or_interpolated": False,
        "source_and_derived_full_torus_plane_counts_match": source_planes_full_torus == derived_planes_full_torus,
    },
    "manifest_reference_only_not_recomputed": {
        "axis_Bphi_T_from_source_verification": MANIFEST["axis_Bphi_T_from_source_verification"],
        "edge_ripple_percent_from_source_verification": MANIFEST["edge_ripple_percent_from_source_verification"],
        "edge_ripple_limit_percent": MANIFEST["edge_ripple_limit_percent"],
        "maximum_centerline_spacing_error_rad": MANIFEST["maximum_centerline_spacing_error_rad"],
    },
    "scope": MANIFEST["scope"],
}

report_path = OUTPUT_DIR / "16fieldperiod_mgrid_conversion_run_report.json"
report_path.write_text(json.dumps(report, indent=2) + "\n", encoding="utf-8")
print("\nFINAL RESULT:", report["status"])
print("Generated MGRID:", output_path)
print("Run report:", report_path)
print("Reference-only source verification values:")
print(f"  Axis Bphi: {MANIFEST['axis_Bphi_T_from_source_verification']:.9f} T")
print(f"  Edge ripple: {MANIFEST['edge_ripple_percent_from_source_verification']:.9f}% (limit {MANIFEST['edge_ripple_limit_percent']:.2f}%)")


## Interpreting the result

`PASS_EXACT_16_FIELD_PERIOD_REPRESENTATION` means the NetCDF source had the expected 48-period metadata, and every generated `br`, `bp`, and `bz` array was exactly equal to the source planes repeated three times. If an existing derived file was attached, it also passed the same exact array comparison.

The reported 16.0000000987 T axis value and 0.75564055% edge ripple are manifest references from the prior source verification, not newly measured by this notebook. This run verifies representation equivalence; it does not establish plasma equilibrium, MHD stability, coil engineering qualification, or reactor performance.
